<h1><center>✨ 𝙏𝙚𝙡𝙚𝙜𝙧𝙖𝙢 𝙍𝙚𝙣𝙖𝙢𝙚𝙧 𝘽𝙤𝙩 — 𝙃𝙀𝙍𝙊𝙆𝙐 𝘿𝙀𝙋𝙇𝙊𝙔</center></h1>

---

### ***Google Colab Deploy Details***
- 🔗 **Repo :** https://github.com/KunalDahal/Telegram-Rename-Bot
- 🚀 **Deploy Kit :** _v1.0_
- ☢️ **Colab Version :** _v1.0_

---
### ***Deploy your Telegram Renamer Bot on Heroku using Google Colab***

> Run every cell from top to bottom. Never publish this notebook after filling in credentials.
>
> One `BOT_TOKEN` = one Heroku worker. To host multiple bots, use a different token and Mongo DB name for every app.

In [ ]:
#@title <center><h3>***Heroku Login***</h3></center><br>

#@markdown ---

Heroku_Email = "" #@param {type:"string"}
Heroku_API_Key = "" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Get your API key from dashboard.heroku.com/account. It stays only in this notebook runtime.</i> )</h6>

#@markdown ---

import shutil, sys, subprocess

if not shutil.which('heroku'):
    !curl -fsSL https://cli-assets.heroku.com/install.sh | sh

from IPython.display import HTML, clear_output, display
clear_output()
display(HTML("<marquee><b>Heroku CLI Installed !</b></marquee>"))

if not all([Heroku_Email, Heroku_API_Key]):
    raise ValueError("Please fill in your Heroku email and API key.")

from os import path as ospath, chmod

netrc_path = ospath.expanduser("~/.netrc")
netrc_creds = f'''machine api.heroku.com
  login {Heroku_Email}
  password {Heroku_API_Key}
machine git.heroku.com
  login {Heroku_Email}
  password {Heroku_API_Key}'''

with open(netrc_path, "w") as netrc_file:
    netrc_file.write(netrc_creds)
chmod(netrc_path, 0o600)

import os
os.environ['HEROKU_API_KEY'] = Heroku_API_Key

display(HTML("<marquee><b>Heroku is connected for this notebook runtime!</b></marquee>"))

In [ ]:
#@title <center><h3>***Create Heroku App(s)***</h3></center><br>

#@markdown ---

App_Names = "" #@param {type:"string"}
#@markdown <h6>( <b>Syntax:</b> <i>my-renamer-one my-renamer-two, separated by space !</i> )</h6>

Server_Region = "us" #@param ["us", "eu"] {allow-input: true}
HK_Team_Name = "" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Optional — only needed if deploying to a Heroku Team.</i> )</h6>

#@markdown ---

if not App_Names.split():
    raise ValueError("Enter at least one app name.")

HK_Team_Flag = f"--team {HK_Team_Name}" if HK_Team_Name else ""
for App_Name in App_Names.split():
    !heroku create $App_Name --region $Server_Region --stack container $HK_Team_Flag

from IPython.display import HTML, display
display(HTML("<marquee><b>App creation complete. Configure every app below!</b></marquee>"))

In [ ]:
#@title <center><h3>***Configure Bot***</h3></center><br>

#@markdown ---
#@markdown Secrets are sent directly to Heroku config vars; they are not placed in Git.
#@markdown Use a unique Mongo DB name per bot token, e.g. `renamer_bot_1`, `renamer_bot_2`. Leave DC blank to accept files from all Telegram data centers.

App_Name = "" #@param {type:"string"}
Bot_Token = "" #@param {type:"string"}
API_ID = 0 #@param {type:"integer"}
API_HASH = "" #@param {type:"string"}
Owner_IDs = "" #@param {type:"string"}
Allowed_Group_IDs = "" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Comma-separated group chat IDs where rename commands are allowed, e.g. -1001234567890. DMs with the bot always work regardless of this setting.</i> )</h6>
Mongo_URI = "" #@param {type:"string"}
Mongo_DB_Name = "renamer_bot" #@param {type:"string"}
Bot_Dump_Chat_ID = "" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Bot-side dump chat for archived uploads. Numeric -100... ID or public @username only — invite links aren't supported here.</i> )</h6>

#@markdown ---
#@markdown ### Optional
Session_String = "" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Telegram Premium user-session string for downloads above 2 GiB. Leave empty for normal bot downloads.</i> )</h6>
Bot_Session_String = "" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Optional persistent Pyrogram session string for the bot client itself, so its peer cache survives dyno restarts.</i> )</h6>
Dump_Chat_ID = "" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Dump chat used by the Premium user session (SESSION_STRING) for files above 2 GiB. Numeric ID, public username, or invite link.</i> )</h6>
Sub_Bot_Tokens = "" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Comma-separated extra bot tokens from @BotFather, used as helper clients that share the download/upload load with the main bot. Leave blank to use only the main bot.</i> )</h6>
DC_Filter = "" #@param {type:"string"}

#@markdown ---
#@markdown ### Concurrency & Suffix
Workers = 4 #@param {type:"integer"}
#@markdown <h6>( <b>Note:</b> <i>Single concurrency knob for the whole bot. Caps how many complete rename jobs run at once, and the same number caps simultaneous downloads and uploads.</i> )</h6>
Upload_Part_Workers = 16 #@param {type:"integer"}
#@markdown <h6>( <b>Note:</b> <i>Number of Telegram file parts uploaded concurrently per individual upload.</i> )</h6>
Command_Suffix = "0" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Digits added to every command. Use 0 for /rename, 2 for /rename2, etc.</i> )</h6>

#@markdown ---

required = {
    'Heroku app': App_Name, 'Bot token': Bot_Token, 'API ID': API_ID, 'API hash': API_HASH,
    'Owner IDs': Owner_IDs, 'Allowed group IDs': Allowed_Group_IDs, 'Mongo URI': Mongo_URI,
    'Mongo DB': Mongo_DB_Name, 'Bot dump chat ID': Bot_Dump_Chat_ID,
}
missing = [label for label, value in required.items() if not str(value).strip() or value == 0]
if missing:
    raise ValueError('Fill in: ' + ', '.join(missing))
if not Command_Suffix.isdigit():
    raise ValueError('Command suffix must contain only digits; use 0 for no suffix.')

config = {
    'BOT_TOKEN': Bot_Token,
    'API_ID': str(API_ID),
    'API_HASH': API_HASH,
    'ALLOWED_GROUP_IDS': Allowed_Group_IDs.strip(),
    'OWNER_IDS': Owner_IDs.strip(),
    'MONGO_URI': Mongo_URI,
    'MONGO_DB_NAME': Mongo_DB_Name.strip(),
    'BOT_SESSION_STRING': Bot_Session_String.strip(),
    'SESSION_STRING': Session_String.strip(),
    'DUMP_CHAT_ID': Dump_Chat_ID.strip(),
    'BOT_DUMP_CHAT_ID': Bot_Dump_Chat_ID.strip(),
    'SUB_BOT_TOKENS': Sub_Bot_Tokens.strip(),
    'DC': DC_Filter.strip(),
    'WORKERS': str(Workers),
    'UPLOAD_PART_WORKERS': str(Upload_Part_Workers),
    'COMMAND_POSTFIX': Command_Suffix,
}

import subprocess
config_args = [f'{key}={value}' for key, value in config.items()]
result = subprocess.run(['heroku', 'config:set', '--app', App_Name.strip(), *config_args], text=True, capture_output=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError('heroku config:set failed')

from IPython.display import HTML, display
display(HTML(f"<marquee><b>Configuration saved for {App_Name.strip()}!</b></marquee>"))

In [ ]:
#@title <center><h3>***Send .env Directly***</h3></center><br>

#@markdown ---
#@markdown Upload a `.env` file (`KEY=VALUE` per line, blank lines and `#` comments are skipped) and push every variable straight to this app's Heroku config vars. Use this instead of the form above if you already have a `.env` file ready.

App_Name = "" #@param {type:"string"}

#@markdown ---

from IPython.display import HTML, display

if not App_Name.strip():
    raise ValueError("Enter the Heroku app name above, then run this cell again.")

def _parse_env(raw_text):
    parsed = {}
    for line in raw_text.splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, _, value = line.partition('=')
        key = key.strip()
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in ('"', "'"):
            value = value[1:-1]
        if key:
            parsed[key] = value
    return parsed

try:
    from google.colab import files
except ImportError:
    raise RuntimeError("Uploading a file needs Google Colab. In plain Jupyter, use the form-based 'Configure Bot' cell instead.")

print("Choose your .env file:")
uploaded = files.upload()
if not uploaded:
    raise ValueError("No file selected.")

filename, raw_bytes = next(iter(uploaded.items()))
env_vars = _parse_env(raw_bytes.decode("utf-8", errors="replace"))

if not env_vars:
    raise ValueError(f"No KEY=VALUE pairs found in {filename}.")

recommended = ['BOT_TOKEN', 'API_ID', 'API_HASH', 'OWNER_IDS', 'ALLOWED_GROUP_IDS', 'MONGO_URI', 'BOT_DUMP_CHAT_ID']
missing = [key for key in recommended if not env_vars.get(key, '').strip()]
if missing:
    display(HTML(f"<b>Warning:</b> {filename} is missing {', '.join(missing)}. Uploading anyway, but the bot may fail to start."))

import subprocess
config_args = [f"{key}={value}" for key, value in env_vars.items()]
result = subprocess.run(['heroku', 'config:set', '--app', App_Name.strip(), *config_args], text=True, capture_output=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError('heroku config:set failed')

display(HTML(f"<marquee><b>{len(env_vars)} variables from {filename} pushed to {App_Name.strip()}!</b></marquee>"))

In [ ]:
#@title <center><h3>***Deploy Bot***</h3></center><br>

#@markdown ---
#@markdown The repository is pushed to every named app, then the worker is scaled to exactly **one** dyno at the size you pick. Configure each app before deploying.

Apps_To_Deploy = "" #@param {type:"string"}
Repository = "https://github.com/KunalDahal/Telegram-Rename-Bot.git" #@param {type:"string"}
Git_Branch = "master" #@param {type:"string"}
Dyno_Size = "basic" #@param ["eco", "basic", "standard-1x", "standard-2x", "performance-m", "performance-l"]
#@markdown <h6>( <b>Note:</b> <i>Dyno count always stays at 1 — Telegram long-polling does not allow two running workers for the same BOT_TOKEN, so this only changes the size/power of that single dyno, not how many run. Eco dynos can sleep after inactivity, which isn't ideal for an always-on bot; Basic or Standard-1X is the practical minimum for continuous polling. Use Standard-2X or a Performance size for heavier ffmpeg/rename workloads.</i> )</h6>

#@markdown ---

names = Apps_To_Deploy.split()
if not names:
    raise ValueError('Enter at least one configured app name.')

from pathlib import Path
repo = Path.cwd() / 'telegram-renamer-bot'
if repo.exists():
    !git -C {repo} fetch origin {Git_Branch}
    !git -C {repo} checkout -B deploy origin/{Git_Branch}
else:
    !git clone --branch {Git_Branch} --single-branch {Repository} {repo}

for App_Name in names:
    !heroku stack:set container --app {App_Name}
    !cd {repo} && git push --force https://git.heroku.com/{App_Name}.git HEAD:main
    !heroku ps:scale worker=1:{Dyno_Size} --app {App_Name}

from IPython.display import HTML, display
display(HTML(f"<marquee><b>Deployment submitted, worker scaled to 1x {Dyno_Size}. Use the Logs form below to watch startup!</b></marquee>"))

In [ ]:
#@title <center><h3>***Resize Worker Dyno***</h3></center><br>

#@markdown ---
#@markdown Change the size of the already-running worker dyno without redeploying any code. Still always exactly 1 dyno per app.

Apps_To_Resize = "" #@param {type:"string"}
New_Dyno_Size = "basic" #@param ["eco", "basic", "standard-1x", "standard-2x", "performance-m", "performance-l"]

#@markdown ---

names = Apps_To_Resize.split()
if not names:
    raise ValueError('Enter at least one app name.')

for App_Name in names:
    !heroku ps:scale worker=1:{New_Dyno_Size} --app {App_Name}

from IPython.display import HTML, display
display(HTML(f"<marquee><b>Worker resized to 1x {New_Dyno_Size}!</b></marquee>"))

In [ ]:
#@title <center><h3>***Show Live Logs***</h3></center><br>

#@markdown ---
Heroku_App = "" #@param {type:"string"}
#@markdown ---

!heroku logs --tail --app {Heroku_App}